# IAQ CO2 forecasting - corrected, leak-free pipeline
Run the single cell below. Set `DATA_PATH` first. Set `IAQ_QUICK=1` (or `QUICK = True`) for a fast smoke test only.

**Read section 1 of the output first**: it prints the timestamp parse, the CO2 autocorrelation in model order (should be high for a real time series) and the number of contiguous segments.

In [1]:
# ==============================================================================
# IoT Indoor Air Quality - CO2 forecasting: corrected, leak-free pipeline
# One cell. Edit DATA_PATH (and QUICK for a fast smoke test), then Run All.
#
# FORMULATION (stated once, used everywhere)
#   At forecast origin t we know: every sensor reading at t, CO2 at t, and CO2 history <= t.
#   Target: CO2 at t + HORIZON steps (strictly in the future).  => no feature can contain the target.
#   Persistence baseline = CO2(t).   All models are scored on the SAME test rows.
# ==============================================================================
import os, re, math, random, logging, warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

# ------------------------------- CONFIG ---------------------------------------
DATA_PATH   = os.environ.get("IAQ_DATA_PATH", r"C:\Users\Mero\OneDrive\Documents\data collection\IoT_Indoor_Air_Quality_Dataset.csv")
QUICK       = bool(int(os.environ.get("IAQ_QUICK", "0")))   # 1 = smaller searches / fewer epochs (smoke test only)
SEED        = 42
HORIZON     = 1                 # steps ahead (1 step = one sampling interval)
LAGS        = [1, 3, 6, 12]     # CO2 lags (rows back from t, within a contiguous segment)
ROLL        = [3, 6, 12]        # rolling windows ending AT t (past + current, never future)
TEST_FRAC   = 0.20
OUTDOOR_PPM = 400.0
HIGH_PPM    = 1000.0
REMOVE_OUTLIERS = False         # Mahalanobis audit is always reported; removal is opt-in
SARIMA_CONTEXT  = 2000
LSTM_WINDOW     = 12

random.seed(SEED); np.random.seed(SEED)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
log = logging.getLogger("iaq")
t0 = time.time()

def rmse(y, p): return float(np.sqrt(mean_squared_error(y, p)))
def banner(s): print("\n" + "=" * 78 + f"\n{s}\n" + "=" * 78)

# --------------------- 1. LOAD + TIMESTAMP DIAGNOSTICS ------------------------
banner("1. DATA LOADING AND TIMESTAMP CHECKS")
raw = pd.read_csv(DATA_PATH)
norm = lambda c: re.sub(r"[^a-z0-9]", "", c.lower())
def find_col(include, exclude=()):
    for c in raw.columns:
        n = norm(c)
        if all(i in n for i in include) and not any(e in n for e in exclude):
            return c
    return None

TS_COL  = next(c for c in raw.columns if "time" in c.lower())
CO2     = find_col(["co2"])
OCC     = find_col(["occupancy"])
MOTION  = find_col(["motion"])
TEMP    = find_col(["temperature"])
HUM     = find_col(["humidity"])
assert CO2 is not None, "No CO2 column found"

def parse_timestamps(s):
    best = None
    for dayfirst in (False, True):
        t = pd.to_datetime(s, dayfirst=dayfirst, errors="coerce")
        nat = int(t.isna().sum())
        mono = float((t.diff().dropna() >= pd.Timedelta(0)).mean())
        print(f"  dayfirst={dayfirst!s:5} | unparseable={nat:6d} | share of non-decreasing steps={mono:.4f}")
        key = (-nat, mono)
        if best is None or key > best[0]:
            best = (key, dayfirst, t)
    print(f"  -> using dayfirst={best[1]}")
    return best[2]

df = raw.copy()
df["Timestamp"] = parse_timestamps(raw[TS_COL])
df = df[df["Timestamp"].notna()].copy()
file_order_ok = (df["Timestamp"].diff().dropna() >= pd.Timedelta(0)).mean() >= 0.999
print(f"  autocorr(1) of CO2 in FILE order: {df[CO2].autocorr(1):.4f}")
if not file_order_ok:
    print("  file is not time-ordered -> stable sort by Timestamp")
    df = df.sort_values("Timestamp", kind="stable")
df = df.reset_index(drop=True)
print(f"  autocorr(1) of CO2 in MODEL order: {df[CO2].autocorr(1):.4f}   <-- must be high for a real time series")

dt_s  = df["Timestamp"].diff().dt.total_seconds()
STEP  = float(dt_s.median())
DT_H  = STEP / 3600.0
df["seg"] = ((dt_s.isna()) | (dt_s > 3 * STEP) | (dt_s <= 0)).cumsum()
print(f"  rows={len(df)} | median step={STEP:.0f}s | contiguous segments={df['seg'].nunique()} | "
      f"CO2 range = {df[CO2].min():.1f}..{df[CO2].max():.1f} ppm")

# --------------------- 2. FEATURES (forecast-origin formulation) --------------
banner("2. FEATURES AND TARGET")
g = df.groupby("seg")[CO2]
for l in LAGS:
    df[f"co2_lag_{l}"] = g.shift(l)
for w in ROLL:
    df[f"co2_rollmean_{w}"] = g.transform(lambda s, w=w: s.rolling(w, min_periods=w).mean())
    df[f"co2_rollstd_{w}"]  = g.transform(lambda s, w=w: s.rolling(w, min_periods=w).std())
df["co2_diff_1"] = df[CO2] - df["co2_lag_1"]
df["target"]     = g.shift(-HORIZON)                    # strictly future value

df["hour_sin"] = np.sin(2 * np.pi * df["Timestamp"].dt.hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["Timestamp"].dt.hour / 24)
df["dayofweek"] = df["Timestamp"].dt.dayofweek
if TEMP and HUM:
    df["temp_x_hum"] = df[TEMP] * df[HUM]

engineered = {"Timestamp", "seg", "target", "hour_sin", "hour_cos", "dayofweek", "temp_x_hum", "co2_diff_1"} | \
             {c for c in df.columns if c.startswith("co2_lag_") or c.startswith("co2_roll")}
exog = [c for c in df.columns if c not in engineered and c not in (TS_COL, CO2)
        and pd.api.types.is_numeric_dtype(df[c])]
CAL      = [c for c in ["hour_sin", "hour_cos", "dayofweek", "temp_x_hum"] if c in df.columns]
SENSORS  = exog + CAL                                     # no CO2 information at all
CO2_NOW  = [CO2]
CO2_HIST = [c for c in df.columns if c.startswith("co2_lag_") or c.startswith("co2_roll")] + ["co2_diff_1"]
FULL     = SENSORS + CO2_NOW + CO2_HIST
print("  sensor/calendar features :", SENSORS)
print("  CO2 now + history        :", CO2_NOW + CO2_HIST)

m = df.dropna(subset=FULL + ["target"]).reset_index(drop=True)
split = int(len(m) * (1 - TEST_FRAC))
train_idx = np.arange(0, split - HORIZON)                # embargo: no train target reaches the test period
test_idx  = np.arange(split, len(m))
Xtr, ytr = m.loc[train_idx, FULL], m.loc[train_idx, "target"].values
Xte, yte = m.loc[test_idx,  FULL], m.loc[test_idx,  "target"].values
print(f"  modelling rows={len(m)} | train={len(Xtr)} | test={len(Xte)} | horizon={HORIZON} step(s) = {HORIZON*STEP/60:.1f} min")

# audits (reported; nothing is silently dropped)
mu = Xtr.values.mean(0); cov = np.cov(Xtr.values, rowvar=False); icov = np.linalg.pinv(cov)
d2 = np.einsum("ij,jk,ik->i", Xtr.values - mu, icov, Xtr.values - mu)
thr = stats.chi2.ppf(0.999, df=Xtr.shape[1])
flag = d2 > thr
print(f"  Mahalanobis audit: {int(flag.sum())} of {len(Xtr)} training rows exceed chi2(0.999) ({100*flag.mean():.2f}%)"
      f" -> {'REMOVED' if REMOVE_OUTLIERS else 'kept (REMOVE_OUTLIERS=False)'}")
if REMOVE_OUTLIERS:
    keep = ~flag; Xtr, ytr = Xtr[keep], ytr[keep]
cs = Xtr[SENSORS].corr().abs() if len(SENSORS) > 1 else pd.DataFrame()
pairs = [(a, b, round(float(cs.loc[a, b]), 3)) for i, a in enumerate(cs.columns) for b in cs.columns[i+1:] if cs.loc[a, b] > 0.95]
print(f"  sensor-pair correlations > 0.95: {pairs if pairs else 'none'}   (CO2 history is never filtered: it is correlated by construction)")

# --------------------- 3. METRICS HELPERS -------------------------------------
results, preds = [], {}
def record(name, p, y=None):
    y = yte if y is None else y
    results.append({"Model": name, "R2": r2_score(y, p), "RMSE_ppm": rmse(y, p), "MAE_ppm": float(mean_absolute_error(y, p))})
    preds[name] = np.asarray(p)
    print(f"  {name:34s} R2={results[-1]['R2']:8.4f}  RMSE={results[-1]['RMSE_ppm']:8.2f}  MAE={results[-1]['MAE_ppm']:8.2f}")

banner("3. BASELINES AND MACHINE-LEARNING MODELS (same features, same test rows)")
record("Lag-1 Persistence (CO2 now)", Xte[CO2].values)

lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression()).fit(Xtr, ytr)
record("Linear Regression", lr.predict(Xte))

dtree = DecisionTreeRegressor(max_depth=10, min_samples_leaf=5, random_state=SEED).fit(Xtr, ytr)
record("Decision Tree", dtree.predict(Xte))

rf_default = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
record("Random Forest (default)", rf_default.predict(Xte))

tscv = TimeSeriesSplit(n_splits=5)
search = RandomizedSearchCV(
    RandomForestRegressor(random_state=SEED, n_jobs=-1),
    param_distributions={"n_estimators": [150, 250], "max_depth": [10, 15, None],
                         "min_samples_split": [2, 5], "min_samples_leaf": [1, 2, 5],
                         "max_features": ["sqrt", 0.5, 1.0]},
    n_iter=3 if QUICK else 8, cv=tscv, scoring="r2", random_state=SEED, n_jobs=1, refit=True)
search.fit(Xtr, ytr)
rf_tuned = search.best_estimator_
print(f"  RandomizedSearchCV(cv=TimeSeriesSplit(5), n_iter={search.n_iter}) best params: {search.best_params_}")
record("Random Forest (tuned)", rf_tuned.predict(Xte))

XGB_PARAMS = dict(n_estimators=100 if QUICK else 300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1)
GB_PARAMS  = dict(n_estimators=100 if QUICK else 300, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=SEED)
xgb = XGBRegressor(**XGB_PARAMS).fit(Xtr, ytr);   record("XGBoost", xgb.predict(Xte))
gbm = GradientBoostingRegressor(**GB_PARAMS).fit(Xtr, ytr); record("Gradient Boosting", gbm.predict(Xte))

# ---- SARIMA: rolling ONE-STEP forecasts (multi-step flat forecasts are not comparable) ----
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    endog = m[CO2].values.astype(float)
    n_ctx = min(SARIMA_CONTEXT, split)
    fit_part = endog[split - n_ctx:split]
    res0 = SARIMAX(fit_part, order=(1, 1, 1)).fit(disp=False)
    ext = res0.append(endog[split:], refit=False)
    sp = np.asarray(ext.get_prediction(start=n_ctx, end=n_ctx + (len(endog) - split) - 1).predicted_mean)
    truth_s = endog[split:]
    pers_s  = endog[split - 1:len(endog) - 1]
    results.append({"Model": "SARIMA(1,1,1) one-step", "R2": r2_score(truth_s, sp), "RMSE_ppm": rmse(truth_s, sp), "MAE_ppm": float(mean_absolute_error(truth_s, sp))})
    print(f"  {'SARIMA(1,1,1) one-step':34s} R2={results[-1]['R2']:8.4f}  RMSE={results[-1]['RMSE_ppm']:8.2f}  (own aligned basis; persistence on same basis R2={r2_score(truth_s, pers_s):.4f})")
except Exception as e:
    log.warning(f"SARIMA skipped: {e}")

# ---- LSTM on real sequences of length LSTM_WINDOW ----
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    tf.random.set_seed(SEED)
    sc = StandardScaler().fit(Xtr.values)
    Xall = sc.transform(m[FULL].values)
    ymu, ysd = ytr.mean(), ytr.std()
    yall = (m["target"].values - ymu) / ysd
    def windows(idx):
        idx = np.array([i for i in idx if i >= LSTM_WINDOW - 1])
        return np.stack([Xall[i - LSTM_WINDOW + 1:i + 1] for i in idx]), yall[idx], idx
    Xw_tr, yw_tr, _ = windows(train_idx)
    Xw_te, yw_te, idx_te_w = windows(test_idx)
    net = Sequential([Input(shape=(LSTM_WINDOW, Xall.shape[1])), LSTM(32), Dropout(0.2), Dense(1)])
    net.compile(optimizer="adam", loss="mse")
    net.fit(Xw_tr, yw_tr, epochs=3 if QUICK else 25, batch_size=128, validation_split=0.1, verbose=0,
            callbacks=[EarlyStopping(patience=4, restore_best_weights=True)])
    lp = net.predict(Xw_te, verbose=0).ravel() * ysd + ymu
    ytrue = yw_te * ysd + ymu
    results.append({"Model": "LSTM (12-step window)", "R2": r2_score(ytrue, lp), "RMSE_ppm": rmse(ytrue, lp), "MAE_ppm": float(mean_absolute_error(ytrue, lp))})
    print(f"  {'LSTM (12-step window)':34s} R2={results[-1]['R2']:8.4f}  RMSE={results[-1]['RMSE_ppm']:8.2f}")
except ImportError:
    log.warning("TensorFlow not installed: LSTM skipped")

# --------------------- 4. PHYSICS MODEL + HYBRID ------------------------------
banner("4. PHYSICS-ONLY AND PHYSICS-INFORMED HYBRID")
PROXY = OCC if OCC else MOTION
def phys_inputs(frame):
    proxy = frame[PROXY].values.astype(float) if PROXY else np.zeros(len(frame))
    return frame[CO2].values.astype(float), proxy
LAM_BOUNDS = (0.0, 6.0)
def fit_physics(frame, y_next):
    c, p = phys_inputs(frame); dt = DT_H * HORIZON
    def loss(q): return np.mean((y_next - (c + dt * (q[0] * p - q[1] * (c - OUTDOOR_PPM)))) ** 2)
    r = minimize(loss, x0=[50.0, 0.5], bounds=[(0.0, None), LAM_BOUNDS], method="L-BFGS-B")
    return r.x
def physics_pred(frame, params):
    c, p = phys_inputs(frame); dt = DT_H * HORIZON
    return c + dt * (params[0] * p - params[1] * (c - OUTDOOR_PPM))
a_hat, lam_hat = fit_physics(Xtr, ytr)
at_bound = abs(lam_hat - LAM_BOUNDS[0]) < 1e-6 or abs(lam_hat - LAM_BOUNDS[1]) < 1e-6
print(f"  fitted generation coeff a={a_hat:.3f} ppm/h per person-proxy | ventilation lambda={lam_hat:.4f} /h"
      f" | at bound: {at_bound}")
phys_te = physics_pred(Xte, (a_hat, lam_hat))
record("Physics-only (fit on 1-step error)", phys_te)

def hybrid_fit_predict(make_res_model, Xa, ya, Xb):
    q = fit_physics(Xa, ya)
    res_model = make_res_model().fit(Xa, ya - physics_pred(Xa, q))
    return physics_pred(Xb, q) + res_model.predict(Xb)
mk_gb = lambda: GradientBoostingRegressor(**GB_PARAMS)
mk_rf = lambda: RandomForestRegressor(n_estimators=150 if not QUICK else 60, max_depth=12, min_samples_leaf=5, random_state=SEED, n_jobs=-1)
record("Hybrid: physics + GB residual", hybrid_fit_predict(mk_gb, Xtr, ytr, Xte))
record("Hybrid: physics + RF residual", hybrid_fit_predict(mk_rf, Xtr, ytr, Xte))

# --------------------- 5. ABLATION (what actually carries the skill) ----------
banner("5. ABLATION BY FEATURE GROUP (Gradient Boosting and Hybrid-GB)")
ablate = []
for label, cols in [("A1 sensors + calendar only", SENSORS),
                    ("A2 + CO2 now", SENSORS + CO2_NOW),
                    ("A3 + CO2 lags/rolling (FULL)", FULL)]:
    p_gb = GradientBoostingRegressor(**GB_PARAMS).fit(Xtr[cols], ytr).predict(Xte[cols])
    q = fit_physics(Xtr, ytr)    # physics needs CO2 now by construction (a physical state variable)
    res = GradientBoostingRegressor(**GB_PARAMS).fit(Xtr[cols], ytr - physics_pred(Xtr, q))
    p_h = physics_pred(Xte, q) + res.predict(Xte[cols])
    ablate.append({"Feature set": label, "GB R2": r2_score(yte, p_gb), "GB RMSE": rmse(yte, p_gb),
                   "Hybrid R2": r2_score(yte, p_h), "Hybrid RMSE": rmse(yte, p_h)})
ablate = pd.DataFrame(ablate); print(ablate.round(4).to_string(index=False))

# --------------------- 6. UNCERTAINTY: quantile + conformal -------------------
banner("6. PREDICTION INTERVALS: coverage, width, conformal calibration (best tree model features = FULL)")
cal_n = int(0.15 * len(Xtr)); fit_n = len(Xtr) - cal_n
Xf, yf, Xc, yc = Xtr.iloc[:fit_n], ytr[:fit_n], Xtr.iloc[fit_n:], ytr[fit_n:]
rows = []
for nominal in (0.80, 0.90, 0.95):
    a = 1 - nominal
    lo_m = GradientBoostingRegressor(loss="quantile", alpha=a / 2, n_estimators=100 if QUICK else 200, max_depth=3, random_state=SEED).fit(Xf, yf)
    hi_m = GradientBoostingRegressor(loss="quantile", alpha=1 - a / 2, n_estimators=100 if QUICK else 200, max_depth=3, random_state=SEED).fit(Xf, yf)
    lo_t, hi_t = lo_m.predict(Xte), hi_m.predict(Xte)
    lo_c, hi_c = lo_m.predict(Xc), hi_m.predict(Xc)
    score = np.maximum(lo_c - yc, yc - hi_c)
    k = min(math.ceil((len(score) + 1) * nominal) / len(score), 1.0)
    qhat = float(np.quantile(score, k))
    for tag, (lo, hi) in {"quantile": (lo_t, hi_t), "conformalized (CQR)": (lo_t - qhat, hi_t + qhat)}.items():
        inside = (yte >= lo) & (yte <= hi)
        rows.append({"Nominal": nominal, "Method": tag, "Coverage": inside.mean(), "MeanWidth_ppm": float(np.mean(hi - lo)),
                     "MedianWidth_ppm": float(np.median(hi - lo))})
intervals = pd.DataFrame(rows); print(intervals.round(4).to_string(index=False))
n_high = int((yte >= HIGH_PPM).sum())
print(f"  test observations >= {HIGH_PPM:.0f} ppm: {n_high} (max in data = {df[CO2].max():.2f} ppm)"
      + ("  -> high-CO2 episodes cannot be evaluated separately in this dataset" if n_high == 0 else ""))
if max(r["R2"] for r in results) < 0.2:
    print("  NOTE: the best model has almost no skill, so the interval is just the marginal CO2 spread and near-nominal "
          "coverage is automatic; judge calibration together with R2 and width.")

# --------------------- 7. FOLD-LEVEL SCORES + SIGNIFICANCE --------------------
banner("7. FOLD-LEVEL R2 (TimeSeriesSplit on the training period) AND TESTS")
def fold_scores(fn):
    out = []
    for tr, va in TimeSeriesSplit(n_splits=5).split(Xtr):
        out.append(r2_score(ytr[va], fn(Xtr.iloc[tr], ytr[tr], Xtr.iloc[va])))
    return np.array(out)
fs = {
    "Persistence":        fold_scores(lambda a, b, c: c[CO2].values),
    "Linear Regression":  fold_scores(lambda a, b, c: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression()).fit(a, b).predict(c)),
    "Random Forest (tuned params)": fold_scores(lambda a, b, c: RandomForestRegressor(**{**search.best_params_, "random_state": SEED, "n_jobs": -1}).fit(a, b).predict(c)),
    "XGBoost":            fold_scores(lambda a, b, c: XGBRegressor(**XGB_PARAMS).fit(a, b).predict(c)),
    "Gradient Boosting":  fold_scores(lambda a, b, c: GradientBoostingRegressor(**GB_PARAMS).fit(a, b).predict(c)),
    "Hybrid (GB residual)": fold_scores(lambda a, b, c: hybrid_fit_predict(mk_gb, a, b, c)),
}
folds = pd.DataFrame(fs); folds.index = [f"fold {i+1}" for i in range(len(folds))]
print(folds.round(4).to_string())
print("  mean +/- SD (ddof=1):", {k: f"{v.mean():.4f} +/- {v.std(ddof=1):.4f}" for k, v in fs.items()})
pairs_to_test = [("Gradient Boosting", "XGBoost"), ("Gradient Boosting", "Random Forest (tuned params)"),
                 ("Gradient Boosting", "Linear Regression"), ("Hybrid (GB residual)", "Gradient Boosting"),
                 ("Gradient Boosting", "Persistence")]
for a, b in pairs_to_test:
    t, p = stats.ttest_rel(fs[a], fs[b]); print(f"  paired t-test {a} vs {b}: t(4)={t:.2f}, p={p:.4f}")
print("  CAUTION: expanding-window folds overlap, so independence is only approximate; treat as indicative.")

def diebold_mariano(y, p1, p2, max_lag=12):
    d = (y - p1) ** 2 - (y - p2) ** 2
    n = len(d); dm = d - d.mean(); var = np.sum(dm * dm) / n
    for k in range(1, max_lag + 1):
        w = 1 - k / (max_lag + 1); var += 2 * w * np.sum(dm[k:] * dm[:-k]) / n
    stat = d.mean() / math.sqrt(var / n)
    return stat, 2 * (1 - stats.norm.cdf(abs(stat)))
for a, b in [("Hybrid: physics + GB residual", "Gradient Boosting"), ("Gradient Boosting", "XGBoost"),
             ("Gradient Boosting", "Lag-1 Persistence (CO2 now)")]:
    s, p = diebold_mariano(yte, preds[a], preds[b]); print(f"  Diebold-Mariano (test errors, Newey-West) {a} vs {b}: DM={s:.2f}, p={p:.4f}")

# --------------------- 8. SHAP (optional) -------------------------------------
banner("8. FEATURE ATTRIBUTION (SHAP, best of GB/XGB/RF by test R2)")
try:
    import shap
    best_name = max(["Gradient Boosting", "XGBoost", "Random Forest (tuned)"], key=lambda n: r2_score(yte, preds[n]))
    best_model = {"Gradient Boosting": gbm, "XGBoost": xgb, "Random Forest (tuned)": rf_tuned}[best_name]
    sample = Xte.sample(min(2000, len(Xte)), random_state=SEED)
    sv = shap.TreeExplainer(best_model).shap_values(sample)
    imp = pd.Series(np.abs(sv).mean(0), index=FULL).sort_values(ascending=False)
    print(f"  model: {best_name}"); print(imp.head(10).round(3).to_string())
    print("  Report these as predictive importance, not causal effects.")
except ImportError:
    log.warning("shap not installed (pip install shap): attribution skipped")
    imp = pd.Series(getattr(gbm, "feature_importances_", np.zeros(len(FULL))), index=FULL).sort_values(ascending=False)
    print("  fallback: Gradient Boosting impurity importance (top 10)"); print(imp.head(10).round(3).to_string())

# --------------------- 9. SUMMARY + SAVE --------------------------------------
banner("9. FINAL TEST-SET SUMMARY")
summary = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
print(summary.round(4).to_string(index=False))
summary.to_csv("iaq_results_leakfree.csv", index=False); folds.to_csv("iaq_fold_scores.csv")
intervals.to_csv("iaq_intervals.csv", index=False); ablate.to_csv("iaq_ablation.csv", index=False)
print(f"\nSaved: iaq_results_leakfree.csv, iaq_fold_scores.csv, iaq_intervals.csv, iaq_ablation.csv  ({time.time()-t0:.0f}s)")



1. DATA LOADING AND TIMESTAMP CHECKS
  dayfirst=False | unparseable=     0 | share of non-decreasing steps=1.0000
  dayfirst=True  | unparseable=     0 | share of non-decreasing steps=1.0000
  -> using dayfirst=False
  autocorr(1) of CO2 in FILE order: 0.0050
  autocorr(1) of CO2 in MODEL order: 0.0050   <-- must be high for a real time series
  rows=97458 | median step=300s | contiguous segments=2 | CO2 range = 400.0..1000.0 ppm

2. FEATURES AND TARGET
  sensor/calendar features : ['Temperature (?C)', 'Humidity (%)', 'PM2.5 (?g/m?)', 'PM10 (?g/m?)', 'TVOC (ppb)', 'CO (ppm)', 'Light Intensity (lux)', 'Motion Detected', 'Occupancy Count', 'hour_sin', 'hour_cos', 'dayofweek', 'temp_x_hum']
  CO2 now + history        : ['CO2 (ppm)', 'co2_lag_1', 'co2_lag_3', 'co2_lag_6', 'co2_lag_12', 'co2_rollmean_3', 'co2_rollstd_3', 'co2_rollmean_6', 'co2_rollstd_6', 'co2_rollmean_12', 'co2_rollstd_12', 'co2_diff_1']
  modelling rows=61246 | train=48995 | test=12250 | horizon=1 step(s) = 5.0 min
  Mah